In [8]:
!pip install trl>=0.9.0 transformers accelerate bitsandbytes

In [9]:
# from huggingface_hub import notebook_login

# token = '...'
# notebook_login()

In [10]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device_name = torch.cuda.get_device_name(0)

print(f'found device: {device}, {device_name}')

found device: cuda, Tesla T4


In [11]:
import pandas as pd
import numpy as np

np.random.seed(42)

splits = {
    'train': 'sarcasm_detection_shared_task_twitter_training.jsonl',
    'test': 'sarcasm_detection_shared_task_twitter_testing.jsonl'
}

url = "hf://datasets/shiv213/Automatic-Sarcasm-Detection-Twitter/"

df = pd.read_json(url + splits["train"], lines=True)

display(df['label'].value_counts())
df = df[df['label'] == 'SARCASM']

print(f'sarcasm df shape: {df.shape}')

df.head()

label
SARCASM        2500
NOT_SARCASM    2500
Name: count, dtype: int64

sarcasm df shape: (2500, 3)


,label,response,context
0,SARCASM,@USER @USER @USER I don't get this .. obviousl...,[A minor child deserves privacy and should be ...
1,SARCASM,@USER @USER trying to protest about . Talking ...,[@USER @USER Why is he a loser ? He's just a P...
2,SARCASM,@USER @USER @USER He makes an insane about of ...,[Donald J . Trump is guilty as charged . The e...
3,SARCASM,@USER @USER Meanwhile Trump won't even release...,[Jamie Raskin tanked Doug Collins . Collins lo...
4,SARCASM,@USER @USER Pretty Sure the Anti-Lincoln Crowd...,[Man ... y ’ all gone “ both sides ” the apoca...


In [21]:
import re

def preprocess_text(text):
    text = re.sub(r'@USER', '', text)
    text = re.sub(r'<URL>', '', text)
    text = re.sub(r'#\w+', '', text)
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text

def build_input_col(df):
    df = df.copy()
    def build(row):
        context_str = "\n".join([
            f"[{i+1}] {preprocess_text(msg)}" 
            for i, msg in enumerate(row['context'])
        ])
        response = preprocess_text(row['response'])
        text = f"""User:Respond sarcastically to the following conversation.
        {context_str}
        Assistant:
        {response}"""
        return text
    df['text'] = df.apply(build, axis=1)
    return df

df = build_input_col(df)

sample = df.sample(3)
sample['text'].to_list()[0]

"User:Respond sarcastically to the following conversation.\n        [1] thanks for ignoring my earlier tweet . Your customer service is pathetic and to think I'm a very regular flyer\n[2] Hi Steve , so sorry for the delay in our response , how can we help ? Why are you so upset ? Danny\n[3] Manchester to London flight 100 minutes late . This route is constantly late\n        Assistant:\n        The irony of this tweet is amazing . CEO of a company that advertised the fact it is reducing its plastic use and you moan that a PLANE is less than 2 hours late for a flight from Manchester to London ."

In [22]:
from datasets import Dataset

df_test = pd.read_json(url + splits["test"], lines=True)
df_test = df_test[df_test['label'] == 'SARCASM']
df_test = build_input_col(df_test)

df_train = df.copy()

ds_train = Dataset.from_pandas(df_train[['text']],  preserve_index=False)
ds_test = Dataset.from_pandas(df_test[['text']], preserve_index=False)

print(f'train/test size: {len(ds_train)}, {len(ds_test)}')

train/test size: 2500, 900


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# model_name = "facebook/opt-1.3b"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [6]:
#Para escolher o valor de r, que deve ser muito menor que min(d, k)
for name, module in base_model.named_modules():
    if hasattr(module, 'weight'):
            print(f"{name}: {module.weight.shape}")

lora_r = 16
lora_alpha = 2* lora_r
lora_layers = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
lora_dropout = 0.1
lora_bias = 'lora_only'

model.embed_tokens: torch.Size([32000, 2048])
model.layers.0.self_attn.q_proj: torch.Size([2048, 2048])
model.layers.0.self_attn.k_proj: torch.Size([256, 2048])
model.layers.0.self_attn.v_proj: torch.Size([256, 2048])
model.layers.0.self_attn.o_proj: torch.Size([2048, 2048])
model.layers.0.mlp.gate_proj: torch.Size([5632, 2048])
model.layers.0.mlp.up_proj: torch.Size([5632, 2048])
model.layers.0.mlp.down_proj: torch.Size([2048, 5632])
model.layers.0.input_layernorm: torch.Size([2048])
model.layers.0.post_attention_layernorm: torch.Size([2048])
model.layers.1.self_attn.q_proj: torch.Size([2048, 2048])
model.layers.1.self_attn.k_proj: torch.Size([256, 2048])
model.layers.1.self_attn.v_proj: torch.Size([256, 2048])
model.layers.1.self_attn.o_proj: torch.Size([2048, 2048])
model.layers.1.mlp.gate_proj: torch.Size([5632, 2048])
model.layers.1.mlp.up_proj: torch.Size([5632, 2048])
model.layers.1.mlp.down_proj: torch.Size([2048, 5632])
model.layers.1.input_layernorm: torch.Size([2048])
model.

In [13]:
from transformers import TrainingArguments, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map="auto")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [18]:
prompt = """User:Respond sarcastically to the following conversation.
    Why do people spend hours on their phones instead of working?
    Assistant:
    """

inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=True,
        temperature=0.9,
        top_p=0.95,
        repetition_penalty=1.3
    )
    
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)

Why do people spend hours on their phones instead of working? The answer is simple: they love being online. According to a survey by comScore, 26% of all Americans aged between 18 and 49 log in at least twice per day with the aim to surf social media sites such as Facebook or Instagram throughout these sessions (source).
It makes sense when you consider that it only costs about $50 for your average smartphone owner to purchase each month. Additionally,


In [23]:
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model

#Para escolher o valor de r, que deve ser muito menor que min(d, k)
for name, module in base_model.named_modules():
    if hasattr(module, 'weight'):
            print(f"{name}: {module.weight.shape}")

lora_r = 16
lora_alpha = 2* lora_r
lora_layers = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
lora_dropout = 0.1
# lora_bias = 'lora_only'
lora_bias = 'none'

lora_config = LoraConfig(
    r=lora_r, lora_alpha=lora_alpha, lora_dropout=lora_dropout,
    bias=lora_bias, task_type="CAUSAL_LM", target_modules=lora_layers,
)

In [24]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )

ds_train_tok = ds_train.map(tokenize, batched=True)
ds_test_tok = ds_test.map(tokenize, batched=True)

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

In [25]:
training_args = TrainingArguments(
    output_dir=f"./{model_name}-sarcasm-lora",
    num_train_epochs=10,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=50,
    learning_rate=2e-4,
    bf16=False,
    fp16=True,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=base_model,
    peft_config=lora_config,
    train_dataset=ds_train_tok,
    eval_dataset=ds_test_tok,
    args=training_args,
)

for name, param in trainer.model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss
1,0.911732,0.931703
2,0.874847,0.927317
3,0.944209,0.926237


TrainOutput(global_step=471, training_loss=1.0316472529352596, metrics={'train_runtime': 2145.71, 'train_samples_per_second': 3.495, 'train_steps_per_second': 0.22, 'total_flos': 2.393897435136e+16, 'train_loss': 1.0316472529352596})

In [73]:
prompt = """User:Respond sarcasticaly to the following conversation.
    How come you don't enjoy waking up early to take a crowded public bus?
    Assistant:
    """
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=True,
        temperature=0.95,
        top_p=0.95,
        repetition_penalty=1.5
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

User:Respond sarcasticaly to the following conversation.
    How come you don't enjoy waking up early to take a crowded public bus?
    Assistant:
     ! You do when it means sitting in front of your computer all day long .


In [102]:
prompt = """User: Respond sarcastically to the following conversation.
     Why do people spend hours on their phones instead of working?
    Assistant:"""

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=True,
        temperature=0.99,
        top_p=0.95,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id 
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

User: Respond sarcastically to the following conversation.
     Why do people spend hours on their phones instead of working?
    Assistant: The most productive way is in front of a real book !
[sarcasm] Do you ever use your phone while reading ? I'm constantly doing that 😒 ❗ it's so funny because my friends also talk too long . We could all learn some patience . But really ... what are we even watching these videos for anyway ? It just looks weird and makes me feel like someone has a shiv wrapped around them head over there :( :


In [118]:
prompt = """User: Respond sarcastically to the following conversation.
    What do you think about Palmeiras, a Brazilian soccer club that has never won the Club World Cup?
    Assistant:"""

inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = base_model.generate(
        **inputs,
        max_new_tokens=90,
        do_sample=True,
        temperature=0.99,
        top_p=0.95,
        repetition_penalty=1.3,
        pad_token_id=tokenizer.eos_token_id 
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

User: Respond sarcastically to the following conversation.
    What do you think about Palmeiras, a Brazilian soccer club that has never won the Club World Cup?
    Assistant: The only thing Palmieras " win" is their name . They haven't even managed an appearance in FIFA Competitions ! - IQ84
    15 of 30 teams with most appearances have won it so far , including China ' s Shenhua & Russia ’ s Kuban Krasnodar 2/3 ...
